In [1]:
import sys
sys.path.append('..')

In [2]:
from pyboy import PyBoy
from base64 import b64encode
import io
import requests

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama


In [3]:
MODEL_ENDPOINT = 'http://127.0.0.1:8080/v1'

In [4]:
from langchain_openai import ChatOpenAI

class ChatLlamaCpp(ChatOpenAI):
    def _create_chat_result(self, response, generation_info=None):
        result = super()._create_chat_result(response, generation_info)
        choices = (response if isinstance(response, dict)
                   else response.model_dump()).get("choices", [])
        for gen, choice in zip(result.generations, choices):
            if rc := choice.get("message", {}).get("reasoning_content"):
                gen.message.additional_kwargs["reasoning_content"] = rc
        return result


In [5]:
pyboy = PyBoy('./roms/pkmsil.gbc', window='null')

In [6]:
with open('./states/bedroom.state', 'rb') as f:
    pyboy.load_state(f)

In [7]:
def load_state(instance: PyBoy, state_file_loc: str):
    with open(state_file_loc, 'rb') as f:
        instance.load_state(f)    

In [8]:
load_state(pyboy, './states/bedroom.state')

In [9]:
def base_64_encode_image(image):
    f = io.BytesIO()
    pyboy.screen.image.save(f, format='png')
    encoded = b64encode(f.getvalue()).decode('utf8')
    return encoded

In [10]:
# resp = requests.post(MODEL_ENDPOINT, json={'prompt': 'hello world'})

In [11]:
model = ChatLlamaCpp(model='Qwen', base_url=MODEL_ENDPOINT, api_key='nada', reasoning_effort="low")

In [12]:
agent = create_agent( model, system_prompt='you are playing pokemon silver')

In [13]:
msg = HumanMessage(content_blocks=[
    {"type": "text", "text": "What does this screenshot from the game show?"},
    {"type": "image", "base64": base_64_encode_image(pyboy.screen.image), "mime_type": "image/png"},
])

In [14]:
def invoke_agent(agent, prompt: str, img = None):

    message_contents = [
        {"type": "text", "text": prompt}
    ]
    
    if img is not None:
        message_contents.append({"type": "image", "base64": base_64_encode_image(img), "mime_type": "image/png"})


    msg = HumanMessage(content_blocks=message_contents)
    
    return agent.invoke({"messages": [msg]})



In [19]:
navigation_agent_prompt = """
You are an agent who is responsible for navigating around the world of pokemon silver. 
Another agent has determined that the game is currently in a state requiring navigation. At the end of this message you
will see a screenshot of the current screen, and a description of the current goal. 
You should identify and clearly describe the tile which the character should move to.
"""

In [20]:
navigation_agent = create_agent(model, system_prompt=navigation_agent_prompt)

In [22]:
invoke_agent(navigation_agent, "Your current goal is to exit the bedroom via the stairs", pyboy.screen.image)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Your current goal is to exit the bedroom via the stairs'}, {'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhEUgAAAKAAAACQCAYAAACPtWCAAAAIYUlEQVR4nO2dL5fyOhDG595zBXI/AhJZiUQikUgkcmU/AhKJrERWVq5EVq5ceeVK3L2ib3ZDSP+kMzBNeH7n7CnbpLRNHyaTSZP8tVwu/yMAlPhb+wLAawMBAlUgQKAKBAhUgQCBKhAgUAUCBKpAgEAVCBCoAgECVSBAoAoECFSBAIEqECBQBQIEqvzD/YL3DdHn15c3rarnSO9J56J9fi5sAQI+1fk06rj1dt8c3yGkY0lE1J6ujagAF/N5668N6f3pHE6HTVD+fV4+5DpCgQVMiPJ8HJRvs32/25dlGdV1HbzlItoI6ft1I707XYLFfE6L+TzoGC3xET2gFdx380jvTtdgjPiyLBM5N6rgCZAfxjVCfLhW1gi+z/oaixa65SIuQO1qLMb0sdakTwS2te1qAPnOP+SaiqLozdNH9Bbw+21H/1JNRP4H4gtD/Bbuff5Q+sIcQ8MgoRalTyC+qn5oK1yqeh1C9AI01HU9OJ623u7FC3m1/qD6Mjz/9/fq5n8pAXJ9zGeKjyixOCAXzvk/qhX7/Lvdjoqi8G6J6GZfG2PKxxa/lG83lGQsoCbV+USHYhF0TH25t4KG7XpJRVH8bIfii+8NYUxPzLm6TM8H1Hbwz1VAHfiA83NpE9x2vSQi6hTlVHo2QhG3gNrVcN+v0lRnkuc3fbJSmO/zfa93X/Z7Pfb1+V5GcNO1if51rKIo7vyW6nzqdaYlfZ23tw+R72kT8nq7Fxf5VEgiDlidTzcPaMjDco/hnN/4cqv1R+95ifwNFqnW55jr1xS3iADbmv6L+ePTmz

In [27]:
pyboy.screen.image.width, pyboy.screen.image.height 

(160, 144)